# Eigenvector-Based PageRank

This notebook demonstrates how PageRank can be obtained using
the eigenvector corresponding to eigenvalue 1 of the Google matrix.

## Workflow

Graph
→ Transition Matrix
→ Google Matrix
→ Eigenvalues and Eigenvectors
→ Eigenvalue ≈ 1
→ Corresponding Eigenvector
→ Normalize
→ PageRank
→ Verify G × PR ≈ PR

In [9]:
import numpy as np
import sys

sys.path.append("../src")

from eigenvalues import (
    calculate_eigenpairs,
    find_eigenvector_for_one,
    normalize_probability_vector
)

from pagerank import (
    build_transition_matrix,
    build_google_matrix,
    pagerank_by_eigenvector,
    verify_pagerank
)   

## 1. Define the Directed Graph

We use four pages (nodes): A, B, C, and D.

An edge `(A, B)` means that page A contains a link to page B.

In [10]:
nodes = ["A", "B", "C", "D"]

edges = [
    ("A", "B"),
    ("A", "C"),
    ("B", "C"),
    ("C", "A"),
    ("C", "D"),
    ("D", "A")
]

print("Nodes:", nodes)
print("Edges:", edges)

Nodes: ['A', 'B', 'C', 'D']
Edges: [('A', 'B'), ('A', 'C'), ('B', 'C'), ('C', 'A'), ('C', 'D'), ('D', 'A')]


## 2. Build the Transition Matrix

The transition matrix represents the probability of moving from one page
to another by following an outgoing link.

We use a column-stochastic convention:

M[target, source] = probability of moving from source to target.

Therefore, each column of M sums to 1.

In [13]:
M = build_transition_matrix(nodes, edges)

print("Transition Matrix M:\n")
print(M)

print("\nColumn sums:")
print(M.sum(axis=0))

Transition Matrix M:

[[0.  0.  0.5 1. ]
 [0.5 0.  0.  0. ]
 [0.5 1.  0.  0. ]
 [0.  0.  0.5 0. ]]

Column sums:
[1. 1. 1. 1.]


## 3. Build the Google Matrix

The Google matrix incorporates both:

1. Following links between pages.
2. Random teleportation to any page.

The Google matrix is defined as:

G = dM + (1-d)/N

where:

- M is the transition matrix
- d is the damping factor
- N is the number of pages

We use the standard damping factor d = 0.85.

In [16]:
damping_factor = 0.85

G = build_google_matrix(
    M,
    damping_factor=damping_factor
)

print("Google Matrix G:\n")
print(G)

print("\nColumn sums:")
print(G.sum(axis=0))

Google Matrix G:

[[0.0375 0.0375 0.4625 0.8875]
 [0.4625 0.0375 0.0375 0.0375]
 [0.4625 0.8875 0.0375 0.0375]
 [0.0375 0.0375 0.4625 0.0375]]

Column sums:
[1. 1. 1. 1.]


## 4. Calculate Eigenvalues and Eigenvectors

For a square matrix A, an eigenvector v and its corresponding eigenvalue λ
satisfy:

A v = λ v

For PageRank, we seek the eigenvector of the Google matrix corresponding
to the eigenvalue λ = 1.

Therefore:

G PR = PR

or equivalently:

G PR = 1 · PR

In [17]:
eigenvalues, eigenvectors = calculate_eigenpairs(G)

print("Eigenvalues:\n")

for i, value in enumerate(eigenvalues):
    print(f"λ{i + 1} =", value)

Eigenvalues:

λ1 = (1.0000000000000004+0j)
λ2 = (-0.21250000000000005+0.5622221536012255j)
λ3 = (-0.21250000000000005-0.5622221536012255j)
λ4 = (-0.4250000000000001+0j)


## 5. Find the Eigenvalue Corresponding to PageRank

Among the eigenvalues of the Google matrix, we identify the eigenvalue
closest to 1.

The corresponding eigenvector contains the PageRank values before
normalization.

In [18]:
eigenvalue, eigenvector = find_eigenvector_for_one(G)

print("Selected eigenvalue:")
print(eigenvalue)

print("\nCorresponding eigenvector:")
print(eigenvector)

Selected eigenvalue:
(1.0000000000000004+0j)

Corresponding eigenvector:
[-0.62204644 -0.33624132 -0.62204644 -0.33624132]


## 6. Normalize the Eigenvector

The raw eigenvector can have an arbitrary magnitude.

PageRank represents a probability distribution, so its entries must sum
to 1.

Therefore, we normalize the eigenvector:

PR = v / Σv

In [19]:
pagerank = normalize_probability_vector(eigenvector)

print("Normalized PageRank vector:\n")

for node, score in zip(nodes, pagerank):
    print(f"{node}: {score:.6f}")

print("\nSum of PageRank:")
print(np.sum(pagerank))

Normalized PageRank vector:

A: 0.324561
B: 0.175439
C: 0.324561
D: 0.175439

Sum of PageRank:
0.9999999999999999


## 7. Verify the PageRank Eigenvector

A valid PageRank vector should satisfy:

G PR ≈ PR

The small difference from zero is due to floating-point numerical
precision.

In [20]:
error, verified = verify_pagerank(
    G,
    pagerank
)

print("Verification error:")
print(error)

print("\nIs the PageRank eigenvector verified?")
print(verified)

Verification error:
2.7336071744532853e-16

Is the PageRank eigenvector verified?
True


## 8. Final PageRank Ranking

The pages are ranked according to their PageRank scores.

In [21]:
ranking = sorted(
    zip(nodes, pagerank),
    key=lambda x: x[1],
    reverse=True
)

print("PageRank Ranking:\n")

for rank, (node, score) in enumerate(ranking, start=1):
    print(f"{rank}. {node} → {score:.6f}")

PageRank Ranking:

1. A → 0.324561
2. C → 0.324561
3. D → 0.175439
4. B → 0.175439


## Eigenvalue Equation and PageRank

For a square matrix A, an eigenvalue λ and its corresponding eigenvector v satisfy:

A v = λ v

For the Google matrix G, PageRank is the eigenvector corresponding to the eigenvalue λ = 1.

Therefore:

G PR = 1 · PR

or:

G PR = PR

In [22]:
# Verify the eigenvalue equation

left_side = G @ eigenvector
right_side = eigenvalue * eigenvector

print("Gv:")
print(left_side)

print("\nλv:")
print(right_side)

print("\nEigenvalue equation verified:")
print(np.allclose(left_side, right_side))

Gv:
[-0.62204644 -0.33624132 -0.62204644 -0.33624132]

λv:
[-0.62204644+0.j -0.33624132+0.j -0.62204644+0.j -0.33624132+0.j]

Eigenvalue equation verified:
True
